In [2]:
%load_ext autoreload
%autoreload 2
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))

import sympy as sp
import numpy as np
from operators import SPIN1, PAULI, LABELS, LABELS_DIP_REAL, LABELS_FULL, LABELS_REAL, PROJECTORS
import itertools as it
from flint import fmpq

In [ ]:
class Alg:
    def __init__(self, ctx, d):
        self.d = d
        self.ctx = ctx
        self.zero = ctx.from_dict({})

    @staticmethod
    def cmul(x, y):
        a, b = x
        c, e = y
        return (a*c - b*e, a*e + b*c)

    @staticmethod
    def cadd_into(D, key, val, sign=1):
        if key in D.keys():
            re, im = D[key]    
            D[key] = (re + val[0], im + val[1]) if sign > 0 else (re - val[0], im - val[1])
        else:
            D[key] = val if sign > 0 else (-val[0], -val[1])
        return
    
    @staticmethod
    def clean(D):
        return {key: val for key, val in D.items() if not (val[0].is_zero() and val[1].is_zero())}

    def pad(self, A, left, right):
        if left == 1 and right == 1:
            return A
        d = self.d
        out = dict()
        for (r, c), v in A.items():
            for L in it.product(range(d), repeat=left):
                for R in it.product(range(d), repeat=right):
                    out[(L + r + R, L + c + R)] = v
    
    def spar_to_dense(self, A):
        d = self.d
        n = len(next(iter(A))[0])
        out = sp.zeros(d**n, d**n)
        for k, v in A.items():
            rtup, ctup = k
            n = len(rtup)
            r = sum([rtup[i] * d**(n-(i+1)) for i in range(n)])
            c = sum([ctup[i] * d**(n-(i+1)) for i in range(n)])
            out[r, c] = v
        return out

    # a bit suboptimal
    def mul(self, A, B):
        AB = dict()
        for ka, va in A.items():
            for kb, vb in B.items():
                if ka[1] == kb[0]:
                    prod = self.cmul(va, vb)
                    self.cadd_into(AB, (ka[0], kb[1]), prod)
        return self.clean(AB)

    def comm(self, A, B):
        out = self.mul(self, A, B)
        BA = self.mul(self, B, A)
        for k, v in BA.items():
            self.cadd_into(out, k, v, sign=-1)
        return self.clean(out)

    def add(self, A, B, sign=1):
        out = dict(A)
        for k, v in B.items():
            self.cadd_into(out, k, v, sign=sign)
        return out

    def scale(self, v, num, den):
        q = fmpq(num, den)
        return (v[0] * q, v[1]*q)

    # a, b could be inputs as well, if this solution turns out to be ugly
    def ti_comm(self, A, B, N):
        a = len(next(iter(A))[0])
        b = len(next(iter(B))[0])
        out = dict()
        for s in range(-(b-1), a-1):
            low = min(0, s)
            high = max(a, s+b)
            n = high - low
            Ap = self.pad(A, -low, high-a)
            Bp = self.pad(B, s-low, high-s-b)
            C = self.comm(Ap, Bp)
            Cp = self.pad(C, 0, N-n)
            for k, v in Cp.items():
                self.cadd_into(out, k, v, sign=1)
        return self.clean(out)

    def canonical(self, D, n):
        d = self.d
        cl_dict = dict()
        X_cur = D
        for l in range(n, 0, -1):
            if not X_cur:
                break
            r = dict()
            for (rw, cl), v in X_cur.items():
                # this is the sparse partial trace
                if rw[0] == cl[0]:
                    self.cadd_into(r, (rw[1:], cl[1:]), v)
            Y = dict(X_cur)
            for (rw, cl), v in r.items():
                for c in range(d):
                    self.cadd_into(Y, ((c,) + rw, (c,) + cl), self.scale(v, 1, d), -1)
            s = dict()
            for (rw, cl), v in Y.items():
                if rw[-1] == cl[-1]:
                    self.cadd_into(s, (rw[:-1], cl[:-1]), v)
            cl = dict(Y)
            for (rw, cl), v in s.items():
                for c in range(d):
                    self.cadd_into(cl, (rw + (c,), cl + (c,)), self.scale(v, 1, d), -1)
            cl_dict[l] = self.clean(cl)
            X_next = {}
            for (rw, cl), v in r.items():
                self.cadd_into(X_next, (rw, cl), self.scale(v, 1, d))
            for (rw, cl), v in s.items():
                self.cadd_into(X_next, (rw, cl), self.scale(v, 1, d))
            X_cur = self.clean(X_next) if l > 1 else {}
        return cl_dict


In [62]:
d = 2
A = {((0,), (1,)): 1, ((1,), (0,)): 1}
def pad(A, left, right, d=d):
    if left == 0 and right == 0:
        return A
    out = dict()
    for (r, c), v in A.items():
        for L in it.product(range(d), repeat=left):
            for R in it.product(range(d), repeat=right):
                out[(L + r + R, L + c + R)] = v
                # print('üres: ', (L + R, L + R))
                # print('LRr: ', (L + R + r, L + R + c))
                # print('LrR: ', (L + r + R, L + c + R))
    return out

def spar_to_dense(A, d):
    print(next(iter(A)))
    n = len(next(iter(A))[0])
    out = sp.zeros(d**n, d**n)
    for k, v in A.items():
        rtup, ctup = k
        n = len(rtup)
        r = sum([rtup[i] * d**(n-(i+1)) for i in range(n)])
        c = sum([ctup[i] * d**(n-(i+1)) for i in range(n)])
        out[r, c] = v
    return out

left = 1
right = 2
Aprime = pad(A, left, right)
Ap_mat = spar_to_dense(d=d, A=Aprime)
display(Ap_mat)
# print('mat done')
# Aprime_mat = sp.zeros(d**(1+left+right), d**(1+left+right))
# # display(Aprime_mat)
# for k, v in Aprime.items():
#     rtup, ctup = k
#     n = len(rtup)
#     r = sum([rtup[i] * d**(n-(i+1)) for i in range(n)])
#     c = sum([ctup[i] * d**(n-(i+1)) for i in range(n)])
#     # c = sum([d**(n-(i+1)) for i in ctup])
#     # print(rtup, ctup)
#     # print(r, c)
#     Aprime_mat[r, c] = v
# display(Aprime_mat)

((0, 0, 0, 0), (0, 1, 0, 0))


Matrix([
[0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
[0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
[0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0],
[0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0],
[1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
[0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
[0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
[0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0],
[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0],
[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0],
[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1],
[0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0],
[0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0],
[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0],
[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0]])

In [44]:
d = 2
left = 3
right = 2
r, c = (1,), (2,)
for L in it.product(range(d), repeat=left):
    for R in it.product(range(d), repeat=right):
        # print('üres: ', (L + R, L + R))
        # print('LRr: ', (L + R + r, L + R + c))
        # print('LrR: ', (L + r + R, L + c + R))
        pass
